# Deploy a Custom LLM (Qwen2.5-7B-Instruct) on Databricks Model Serving with vLLM

This notebook downloads a HuggingFace LLM, validates it locally in a **Serverless GPU notebook**, then deploys it to **Databricks Model Serving** via the new Serverless Optimized Deployments path backed by vLLM. See [Serve custom LLMs](https://docs.databricks.com/aws/en/machine-learning/model-serving/serve-custom-llms).

**Compute** — Serverless GPU notebook (Databricks AI Runtime), **A10 (24 GB)**.

**Model** — `Qwen/Qwen2.5-7B-Instruct`. Ungated, fits an A10 at `dtype=float16` (~14 GB weights + KV cache). Swap `MODEL_REPO_ID` for any vLLM-compatible model that fits the GPU.

Edit the **Configuration** cell before running the rest of the notebook.

## Set up the environment (Serverless GPU with A10)

In [ ]:
%sh
nvidia-smi

In [ ]:
# Serving runtime requirements. Pinned versions match the Databricks starter.
%pip install vllm==0.11.2 transformers==4.57.6 openai==2.17.0 opencv-python-headless==4.12.* mlflow==3.12.0 hf_transfer==0.1.9
%restart_python

In [ ]:
# /Workspace can't hold multi-GB weights — work from local disk.
import os, tempfile
workdir = tempfile.mkdtemp()
os.chdir(workdir)
print("workdir:", workdir)

## Configuration

All knobs in one place. The block below is split into:
1. **Model + paths** — what to download, where it lands.
2. **vLLM tuning** — required + optional flags (set the optional ones to `None`/`False` to drop them).
3. **Ports** — local test must use 3000–3999 on Serverless GPU; Serving always uses 8080.
4. **Unity Catalog + Endpoint** — where the registered model + endpoint go.

In [ ]:
from databricks.sdk.service.serving import ServingModelWorkloadType

# --- 1) Model + paths -----------------------------------------------------
MODEL_REPO_ID = "Qwen/Qwen2.5-7B-Instruct"
ARTIFACTS_PATH = "qwen25_7b"   # Doubles as the local dir AND the MLflow artifact key — vLLM resolves it correctly in both contexts.
SERVED_MODEL_NAME = "qwen"     # Value clients pass in `model` when hitting vLLM directly.

# --- 2) vLLM tuning -------------------------------------------------------
# Required defaults.
DTYPE = "float16"                  # float16 | bfloat16 | float32 — A10 has no bf16 tensor cores, fp16 is best.
MAX_MODEL_LEN = 16384              # Context window. Larger = more KV cache memory.
GPU_MEMORY_UTILIZATION = 0.85      # Fraction of GPU memory vLLM may use.

# Optional tunables — set to None / False to omit the flag entirely.
ENFORCE_EAGER = False              # True: skip CUDA graph capture. Faster cold start, ~10-20% lower throughput.
TENSOR_PARALLEL_SIZE = 1           # >1 only if the endpoint has multiple GPUs (A10 is single-GPU, leave at 1).
MAX_NUM_SEQS = None                # e.g. 64. Cap on concurrent requests; raises throughput, costs KV memory.
MAX_NUM_BATCHED_TOKENS = None      # e.g. 8192. Throughput vs latency knob for prefill batching.
KV_CACHE_DTYPE = None              # e.g. "fp8". Halves KV cache memory at minor quality cost.
QUANTIZATION = None                # e.g. "fp8" | "awq" | "gptq". Required if the HF repo is pre-quantized.
SWAP_SPACE = None                  # GiB of CPU RAM for KV cache spill (e.g. 4).
EXTRA_VLLM_ARGS = []               # Escape hatch: any other flags, e.g. ["--disable-log-requests"].

# --- 3) Ports -------------------------------------------------------------
LOCAL_PORT = 3080      # Serverless GPU only allows inbound on 3000-3999.
SERVING_PORT = 8080    # Required by Model Serving — do not change.

# --- 4) Unity Catalog + Endpoint -----------------------------------------
UC_MODEL_NAME = "main.<your-schema>.qwen25_7b_instruct"   # catalog.schema.model — must be writable by you.

ENDPOINT_NAME = "qwen25-7b-endpoint"                       # Unique within the workspace.
WORKLOAD_TYPE = ServingModelWorkloadType.GPU_MEDIUM        # T4: GPU_SMALL · A10: GPU_MEDIUM · H100: GPU_XLARGE
WORKLOAD_SIZE = "Small"                                    # Small | Medium | Large
SCALE_TO_ZERO_ENABLED = True                               # Cheap for dev; cold start takes minutes.

## Download the model

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=MODEL_REPO_ID,
    local_dir=ARTIFACTS_PATH,
)

## Test the model in the notebook

The same `entrypoint(port)` string is used twice:
- Locally with `port=LOCAL_PORT` (3080) for validation.
- Stored in MLflow metadata with `port=SERVING_PORT` (8080) so Model Serving knows how to launch vLLM on the fleet.

Because `ARTIFACTS_PATH` is used both as the local directory name **and** as the MLflow artifact key, the literal `--model qwen25_7b` resolves correctly in both contexts.

In [ ]:
def entrypoint(port: int) -> str:
    args = [
        "python", "-u", "-m", "vllm.entrypoints.openai.api_server",
        "--model", ARTIFACTS_PATH,
        "--served-model-name", SERVED_MODEL_NAME,
        "--host", "0.0.0.0",
        "--port", str(port),
        "--dtype", DTYPE,
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--tensor-parallel-size", str(TENSOR_PARALLEL_SIZE),
    ]
    if ENFORCE_EAGER:
        args.append("--enforce-eager")
    if MAX_NUM_SEQS is not None:
        args += ["--max-num-seqs", str(MAX_NUM_SEQS)]
    if MAX_NUM_BATCHED_TOKENS is not None:
        args += ["--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS)]
    if KV_CACHE_DTYPE is not None:
        args += ["--kv-cache-dtype", KV_CACHE_DTYPE]
    if QUANTIZATION is not None:
        args += ["--quantization", QUANTIZATION]
    if SWAP_SPACE is not None:
        args += ["--swap-space", str(SWAP_SPACE)]
    args += EXTRA_VLLM_ARGS
    return " ".join(args)

print(entrypoint(LOCAL_PORT))

In [ ]:
import subprocess

# Start vLLM in the background. Logs go to process.log so the %sh tail in the next cell can wait on readiness.
log = open("process.log", "w")
subprocess.Popen(
    ["bash", "-lc", entrypoint(LOCAL_PORT)],
    stdout=log,
    stderr=subprocess.STDOUT,
    text=True,
    start_new_session=True,
)

In [ ]:
%sh
# Tail logs until vLLM is ready. If this hangs, vLLM startup probably encountered an error — scroll up in process.log.
tail -f process.log | sed -u '/Application startup complete/q'

In [ ]:
import requests

resp = requests.post(
    f"http://localhost:{LOCAL_PORT}/invocations",
    json={"messages": [{"role": "user", "content": "Give me three creative names for a coffee shop near a beach."}]},
)
print(resp.json()["choices"][0]["message"]["content"])

In [ ]:
import requests, json

# vLLM streams completions as SSE: one JSON chunk per `data: ` line, terminated by `data: [DONE]`.
resp = requests.post(
    f"http://localhost:{LOCAL_PORT}/invocations",
    json={"messages": [{"role": "user", "content": "Tell me a story that is about 300 words!"}], "stream": True},
    stream=True,
)

for line in resp.iter_lines():
    if not line:
        continue
    if line == b"data: [DONE]":
        break
    if line.startswith(b"data: "):
        data = json.loads(line[6:])
        delta = data["choices"][0].get("delta", {})
        if "content" in delta:
            print(delta["content"], end="", flush=True)

In [ ]:
%sh
# Free the GPU before logging the model.
pkill -f vllm.entrypoints.openai.api_server

## Log the model with our custom entrypoint

`LLMModel.predict` is a **required placeholder** — Serving runs the `entrypoint` string, not Python. The interesting bits:
- `artifacts={"model_dir": ARTIFACTS_PATH}` bundles the weights into the MLflow model. (Note: the actual flag passed to vLLM is `--model {ARTIFACTS_PATH}`, which resolves to the artifact directory at serve time because the key matches.)
- `metadata["task"] = "llm/v1/chat"` declares this as a chat LLM.
- `metadata["entrypoint"] = entrypoint(SERVING_PORT)` — note **SERVING_PORT (8080)**, not the local 3080.

In [ ]:
import mlflow
from mlflow.pyfunc.model import ChatModel, ChatCompletionResponse

class LLMModel(ChatModel):
    def predict(self, context, messages, params):
        return ChatCompletionResponse.from_dict({"choices": []})

model_info = mlflow.pyfunc.log_model(
    name=SERVED_MODEL_NAME,
    python_model=LLMModel(),
    artifacts={
        "model_dir": ARTIFACTS_PATH,
    },
    metadata={
        "task": "llm/v1/chat",
        "entrypoint": entrypoint(SERVING_PORT),
    },
    extra_pip_requirements=[
        "mlflow==3.12.0",
    ],
)
model_info.model_uri

## Register the model to Unity Catalog

In [ ]:
import mlflow

# env_pack is required. Custom LLM Serving depends on Serverless Optimized Deployments. The endpoint will not work without it.
# https://docs.databricks.com/aws/en/machine-learning/model-serving/serverless-optimized-deployments
model_version = mlflow.register_model(
    model_info.model_uri,
    UC_MODEL_NAME,
    env_pack="databricks_model_serving",
)

## Create an endpoint with the model

| GPU      | `workload_type`  | Memory |
|----------|------------------|--------|
| T4       | `GPU_SMALL`      | 16 GB  |
| **A10**  | **`GPU_MEDIUM`** | **24 GB** |
| H100     | `GPU_XLARGE`     | 80 GB  |

In [ ]:
from datetime import timedelta
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

config = EndpointCoreConfigInput(
    served_entities=[
        ServedEntityInput(
            entity_name=UC_MODEL_NAME,
            entity_version=str(model_version.version),
            workload_type=WORKLOAD_TYPE,
            workload_size=WORKLOAD_SIZE,
            scale_to_zero_enabled=SCALE_TO_ZERO_ENABLED,
        )
    ]
)

w = WorkspaceClient()
w.serving_endpoints.create_and_wait(
    name=ENDPOINT_NAME,
    config=config,
    timeout=timedelta(minutes=30),
)

## Query the ready endpoint

In [ ]:
# Databricks SDK.
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()

resp = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    messages=[ChatMessage(role=ChatMessageRole.USER, content="Hi, what model are you?")],
)
print(resp.choices[0].message.content)

In [ ]:
# OpenAI client.
from openai import OpenAI

DATABRICKS_HOST = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints",
)

response = client.chat.completions.create(
    model=ENDPOINT_NAME,
    messages=[
        {"role": "user", "content": "Explain what vLLM does in 2 sentences."},
    ],
)
print(response.choices[0].message.content)

In [ ]:
# OpenAI client — streaming.
stream = client.chat.completions.create(
    model=ENDPOINT_NAME,
    messages=[
        {"role": "user", "content": "Hello, tell me a 200 word story"},
    ],
    stream=True,
)

for event in stream:
    delta = event.choices[0].delta
    if delta.content:
        print(delta.content, end="", flush=True)